# Графики test predictions

Источник: `./model_test_predictions/*_test_predictions.csv`. Строки с пустым `power_mode_band` не используются. Для каждой пары `модель + vib_XX` строится отдельный PNG: факт, точечный прогноз и интервал `pred_lower`-`pred_upper`.


In [1]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PRED_DIR = Path("model_test_predictions")
OUT_DIR = Path("charts") / "part3_test_predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_USECOLS = [
    "model",
    "window_id",
    "power_mode_band",
    "origin_ts",
    "target_ts",
    "target_name",
    "actual_value",
    "pred_point",
    "pred_lower",
    "pred_upper",
]
VALUE_COLS = ["actual_value", "pred_point", "pred_lower", "pred_upper"]
VIB_TARGETS = [f"vib_{i:02d}" for i in range(1, 15)]
MODEL_ORDER = ["SARIMA", "VAR", "LightGBM", "TCN", "Transformer"]


def _sanitize_filename(name: str, max_len: int = 120) -> str:
    s = re.sub(r"[<>:\"/\\|?*\n\r\t]", "_", name)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_len].rstrip()


def load_test_predictions() -> pd.DataFrame:
    files = sorted(PRED_DIR.glob("*_test_predictions.csv"))
    if not files:
        raise FileNotFoundError(f"В {PRED_DIR.as_posix()} не найдены файлы `*_test_predictions.csv`.")

    frames: list[pd.DataFrame] = []
    for path in files:
        df = pd.read_csv(path, usecols=PRED_USECOLS)
        df["source_file"] = path.name
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)
    data["power_mode_band"] = data["power_mode_band"].astype("string").str.strip()
    invalid_power_mode = (
        data["power_mode_band"].isna()
        | data["power_mode_band"].eq("")
        | data["power_mode_band"].str.lower().isin({"nan", "none", "null", "nat"})
    )
    data = data.loc[~invalid_power_mode].copy()

    data["target_ts"] = pd.to_datetime(data["target_ts"], errors="coerce")
    data["origin_ts"] = pd.to_datetime(data["origin_ts"], errors="coerce")
    for col in VALUE_COLS:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data.dropna(subset=["model", "target_ts", "target_name", *VALUE_COLS])
    data["target_name"] = data["target_name"].astype(str)
    data["model"] = data["model"].astype(str)
    data["window_id"] = data["window_id"].astype(str)
    return data.sort_values(["target_name", "model", "window_id", "target_ts"])


def _ordered_models(data: pd.DataFrame) -> list[str]:
    present = set(data["model"].unique())
    ordered = [m for m in MODEL_ORDER if m in present]
    ordered.extend(sorted(present - set(ordered)))
    return ordered


def plot_model_target_predictions(
    data: pd.DataFrame,
    target_name: str,
    model: str,
    out_dir: Path = OUT_DIR,
) -> Path | None:
    d_model = data.loc[data["target_name"].eq(target_name) & data["model"].eq(model)].copy()
    if d_model.empty:
        return None

    d_model = d_model.sort_values(["window_id", "target_ts"])
    actual = (
        d_model.groupby("target_ts", as_index=False)["actual_value"]
        .mean()
        .sort_values("target_ts")
    )

    fig, ax = plt.subplots(figsize=(18, 6), dpi=140)
    ax.plot(
        actual["target_ts"].to_numpy(),
        actual["actual_value"].to_numpy(),
        color="#222222",
        linewidth=1.25,
        label="Факт",
    )

    first_window = True
    for _, window in d_model.groupby("window_id", sort=False):
        window = window.sort_values("target_ts")
        x_pred = window["target_ts"].to_numpy()
        ax.fill_between(
            x_pred,
            window["pred_lower"].to_numpy(),
            window["pred_upper"].to_numpy(),
            color="#4C78A8",
            alpha=0.22,
            linewidth=0,
            label="Интервал прогноза" if first_window else None,
        )
        ax.plot(
            x_pred,
            window["pred_point"].to_numpy(),
            color="#E45756",
            linewidth=1.0,
            label="Прогноз" if first_window else None,
        )
        first_window = False

    window_count = d_model["window_id"].nunique()
    origin_min = d_model["origin_ts"].min()
    origin_max = d_model["origin_ts"].max()
    if pd.isna(origin_min):
        origin_text = ""
    elif origin_min == origin_max:
        origin_text = f"origin: {origin_min}"
    else:
        origin_text = f"origin: {origin_min} - {origin_max}"

    title_tail = f"окон: {window_count}"
    if origin_text:
        title_tail += f", {origin_text}"

    ax.set_title(f"{model} | {target_name}\nФакт и интервалы предсказаний ({title_tail})")
    ax.set_xlabel("Время")
    ax.set_ylabel(target_name)
    ax.grid(True, linewidth=0.4, alpha=0.3)
    ax.legend(loc="upper right")
    fig.tight_layout()

    model_dir = out_dir / _sanitize_filename(model)
    model_dir.mkdir(parents=True, exist_ok=True)
    out_file = model_dir / f"{_sanitize_filename(target_name)}__{_sanitize_filename(model)}__actual_vs_prediction_interval.png"
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)
    return out_file


def plot_all_targets(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for target_name in VIB_TARGETS:
        for model in _ordered_models(data.loc[data["target_name"].eq(target_name)]):
            model_data = data.loc[data["target_name"].eq(target_name) & data["model"].eq(model)]
            out_file = plot_model_target_predictions(data, target_name, model)
            rows.append({
                "target_name": target_name,
                "model": model,
                "rows": int(len(model_data)),
                "windows": int(model_data["window_id"].nunique()) if not model_data.empty else 0,
                "start": model_data["target_ts"].min() if not model_data.empty else pd.NaT,
                "end": model_data["target_ts"].max() if not model_data.empty else pd.NaT,
                "out": None if out_file is None else out_file.as_posix(),
            })
    return pd.DataFrame(rows)

In [2]:
predictions = load_test_predictions()
report = plot_all_targets(predictions)
report

,target_name,model,rows,windows,start,end,out
0,vib_01,SARIMA,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/SARIMA/vib_01__S...
1,vib_01,VAR,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/VAR/vib_01__VAR_...
2,vib_01,LightGBM,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/LightGBM/vib_01_...
3,vib_01,TCN,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/TCN/vib_01__TCN_...
4,vib_01,Transformer,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/Transformer/vib_...
...,...,...,...,...,...,...,...
65,vib_14,SARIMA,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/SARIMA/vib_14__S...
66,vib_14,VAR,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/VAR/vib_14__VAR_...
67,vib_14,LightGBM,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/LightGBM/vib_14_...
68,vib_14,TCN,30240,1,2025-12-01 00:06:00,2025-12-22 00:05:00,charts/part3_test_predictions/TCN/vib_14__TCN_...
